In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3" #without it gptq will try to start on all gpus
from transformers import AutoModelForCausalLM
import torch
from transformers import BitsAndBytesConfig
from gptqmodel import GPTQModel

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


WARN  Feature `utils/Perplexity` requires Python < 3.14 and Python GIL enabled and Python >= 3.13.3T (T for Threading-Free edition of Python) plus Torch 2.8. Feature is currently skipped/disabled.


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[W715 15:42:54.267749040 Context.cpp:424] Warning: torch.backends.cuda.preferred_linalg_library is an experimental feature. If you see any error or unexpected behavior when this flag is set please file an issue on GitHub. (function operator())


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO:datasets:TensorFlow version 2.21.0 available.


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 5.8.0
Transformers : 5.5.1
Torch        : 2.10.0+cu128
Triton       : 3.6.0


In [2]:
MODEL = "Qwen/Qwen3-30B-A3B-GPTQ-Int4" #dont fit because GPTQ try to repack new tensors in transformers 5 and consumes extramemory
#MODEL = "Qwen/Qwen3-30B-A3B" #dont fit with bitsandbytes, because in transformers 5 it contains experts as one huge Parameter(128, ..., ...), not many small nn.Linear
#MODEL = "Qwen/Qwen3-30B-A3B-FP8" #supports only on 4090/H100
quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,  # Compute in float16 for speed
        bnb_4bit_use_double_quant=True,  # Double quantization for extra memory savings
        # Normalized float 4-bit (optimal for LLMs)
        bnb_4bit_quant_type="nf4",
        llm_int8_skip_modules=[
            "lm_head",
            "mlp.gate", #because DeepSeekV2ForCausalLM has a f.linear(self.gate.weight...) instead of self.gate()
            #it couldn't be substituded with Linear4bit
        ],
    )

In [3]:
model = GPTQModel.load(
    MODEL,
    device=f"cuda:0",
    profile="low_memory",
)

allocated_gib = torch.cuda.memory_allocated(0) / 2**30
reserved_gib = torch.cuda.memory_reserved(0) / 2**30
print(f"Loaded: {allocated_gib:.1f} GiB allocated, {reserved_gib:.1f} GiB reserved")
# model = AutoModelForCausalLM.from_pretrained(MODEL, 
#                                              trust_remote_code=False, #нужно в версии transformers 4, чтобы загружать доп код от дипсика, но в новых уже есть нативая поддержка DeepSeekV2 
#                                                 device_map='cuda:0',
#                                                 # dtype=torch.float16,
#                                                 #quantization_config=quantization_config,
#                                                 # attn_implementation="sdpa", 
#                                                 # kv_cache_dtype="int8"
#                                                 )

INFO:httpx:HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/config.json "HTTP/1.1 200 OK"


from_quantized: adapter: None


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/revision/main "HTTP/1.1 200 OK"


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

INFO  Patched transformers.models.qwen3_moe.modeling_qwen3_moe.Qwen3MoeSparseMoeBlock -> defuser.modeling.unfused_moe.qwen3_moe.LinearQwen3MoeSparseMoeBlock


...

INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}



INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.


INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "temperature": 0.6,
  "top_k": 20,
  "top_p": 0.95
}



INFO  Kernel: loaded -> `[MarlinQuantLinear]`                                  


Loaded: 15.6 GiB allocated, 19.6 GiB reserved


In [4]:
from transformers import Qwen3MoeForCausalLM

Wrapper above model Qwen3MoeQModel from GPTQ  
forward() just forwrads into inner method

In [5]:
model

Qwen3MoeQModel(
  (model): Qwen3MoeForCausalLM(
    (model): Qwen3MoeModel(
      (embed_tokens): Embedding(151936, 2048)
      (layers): ModuleList(
        (0-47): 48 x Qwen3MoeDecoderLayer(
          (self_attn): Qwen3MoeAttention(
            (q_proj): MarlinQuantLinear()
            (k_proj): MarlinQuantLinear()
            (v_proj): MarlinQuantLinear()
            (o_proj): MarlinQuantLinear()
            (q_norm): Qwen3MoeRMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3MoeRMSNorm((128,), eps=1e-06)
            (rotary_fn): Func()
          )
          (mlp): LinearQwen3MoeSparseMoeBlock(
            (gate): Qwen3MoeTopKRouter()
            (experts): ModuleList(
              (0-127): 128 x Qwen3MoeMLP(
                (gate_proj): MarlinQuantLinear()
                (up_proj): MarlinQuantLinear()
                (down_proj): MarlinQuantLinear()
                (act_fn): SiLUActivation()
              )
            )
          )
          (input_layernorm): Qwen3MoeRMSNorm

In [6]:
model.__dict__

{'training': True,
 '_parameters': {},
 '_buffers': {},
 '_non_persistent_buffers_set': set(),
 '_backward_pre_hooks': OrderedDict(),
 '_backward_hooks': OrderedDict(),
 '_is_full_backward_hook': None,
 '_forward_hooks': OrderedDict(),
 '_forward_hooks_with_kwargs': OrderedDict(),
 '_forward_hooks_always_called': OrderedDict(),
 '_forward_pre_hooks': OrderedDict(),
 '_forward_pre_hooks_with_kwargs': OrderedDict(),
 '_state_dict_hooks': OrderedDict(),
 '_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_pre_hooks': OrderedDict(),
 '_load_state_dict_post_hooks': OrderedDict(),
 '_modules': {'model': Qwen3MoeForCausalLM(
    (model): Qwen3MoeModel(
      (embed_tokens): Embedding(151936, 2048)
      (layers): ModuleList(
        (0-47): 48 x Qwen3MoeDecoderLayer(
          (self_attn): Qwen3MoeAttention(
            (q_proj): MarlinQuantLinear()
            (k_proj): MarlinQuantLinear()
            (v_proj): MarlinQuantLinear()
            (o_proj): MarlinQuantLinear()
            

In [7]:
from gptqmodel.models.definitions.qwen3_moe import Qwen3MoeQModel
print(type(model))


<class 'gptqmodel.models.definitions.qwen3_moe.Qwen3MoeQModel'>


In [8]:
import transformer_lens
print(transformer_lens.__file__)

/glazkov-dev/TransformerLens/transformer_lens/__init__.py


In [9]:
from transformer_lens.model_bridge import TransformerBridge

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model.model,
    dtype=torch.float16,
)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:H

In [ ]:
from utils.mmlu_batch_generator import MMLUBatchGenerator
SELECTED_TASKS = ["anatomy",
            # "conceptual_physics",
            # "human_sexuality",
            "machine_learning",
            "management",
            # "marketing",
            # "nutrition",
             "philosophy",
            # "us_foreign_policy",
              "world_religions"]
gen = MMLUBatchGenerator(subjects=SELECTED_TASKS, split='validation', batch_size=16, include_metadata=False)

Initialized MMLU batch generator with 5 subjects
Split: validation, Batch size: 16


In [11]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL,
                                                device_map='cuda:3',
                                                )

for batch in gen:
    inputs = tokenizer(batch, return_tensors="pt",
            padding=True,  # Pad to longest in batch
            truncation=True,
            max_length=512,  # Adjust based on your needs
            add_special_tokens=True
    )  
    inputs = {k: v.to(bridge.device) for k, v in inputs.items()}
    print(inputs.keys())
    bridge_outputs, cache = bridge.run_with_cache(inputs['input_ids'], attention_mask=inputs['attention_mask'])
    print(bridge_outputs)  # Example: (batch_size, sequence_length, vocab_size)
    break  # Remove this break to process all batches

INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:H


📚 Loading subject 1/5: anatomy


INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/cais/mmlu/resolve/c30699e8356da336a370243923dbaf21066bb9fe/mmlu.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/cais/mmlu/cais/mmlu.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/cais/mmlu/revision/c30699e8356da336a370243923dbaf21066bb9fe "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/cais/mmlu/resolve/c30699e8356da336a370243923dbaf21066bb9fe/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=cais/mmlu "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/cais/mmlu/tree/c30699e8356da336a370243923dbaf21066bb9fe/abstract_algebra?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/cais/mmlu/tree/c30699e8356da336a3702

  ✓ Created batch with 14 questions
dict_keys(['input_ids', 'attention_mask'])
tensor([[[ 0.2961,  2.9473,  1.9756,  ..., -2.7500, -2.9902, -2.5273],
         [10.5625, 11.0312, 10.1328,  ...,  3.6621,  4.3789,  4.0586],
         [ 5.3516,  8.2578,  5.3711,  ...,  1.0547,  1.9053,  1.0928],
         ...,
         [15.5312, 22.9219, 17.1406,  ...,  7.2656,  7.7227,  7.4688],
         [16.6406, 22.2188, 17.8438,  ...,  5.1172,  6.0234,  5.2266],
         [16.5469, 22.9688, 19.4688,  ...,  4.7812,  5.1055,  4.6016]],

        [[ 0.2961,  2.9473,  1.9756,  ..., -2.7500, -2.9902, -2.5273],
         [10.5625, 11.0312, 10.1328,  ...,  3.6621,  4.3789,  4.0586],
         [ 5.3516,  8.2578,  5.3711,  ...,  1.0547,  1.9053,  1.0928],
         ...,
         [16.8125, 22.6562, 21.5781,  ...,  7.1211,  7.9727,  7.4180],
         [10.3281,  7.8164,  9.9297,  ..., -2.5508, -2.1523, -2.2598],
         [17.1719, 21.5000, 19.8594,  ...,  5.8516,  6.3711,  6.1992]],

        [[ 5.6055,  7.7617,  7.7695, 

In [19]:
bridge_outputs.shape

torch.Size([14, 33, 151936])

Must be float16 everywhere

In [12]:
hf_model = model.model

print("embedding:", hf_model.model.embed_tokens.weight.dtype)
print(
    "norm:",
    hf_model.model.layers[0].input_layernorm.weight.dtype,
)
print(
    "q scales:",
    hf_model.model.layers[0].self_attn.q_proj.scales.dtype,
)

embedding: torch.float16
norm: torch.float16
q scales: torch.float16


In [13]:
attn = hf_model.model.layers[0].self_attn

print(type(attn))
print(type(attn.q_proj))

<class 'transformer_lens.model_bridge.generalized_components.position_embeddings_attention.PositionEmbeddingsAttentionBridge'>
<class 'transformer_lens.model_bridge.generalized_components.linear.LinearBridge'>


In [14]:
print(type(hf_model.model.layers[0].self_attn))
print(type(hf_model.model.layers[0].self_attn.q))
print(type(hf_model.model.layers[0].self_attn.q.original_component))

<class 'transformer_lens.model_bridge.generalized_components.position_embeddings_attention.PositionEmbeddingsAttentionBridge'>
<class 'transformer_lens.model_bridge.generalized_components.linear.LinearBridge'>
<class 'gptqmodel.nn_modules.qlinear.marlin.MarlinQuantLinear'>


In [15]:
import torch
from collections import defaultdict

hook_handles = []
call_counts = defaultdict(int)

def tensor_info(value):
    if isinstance(value, torch.Tensor):
        return {
            "shape": tuple(value.shape),
            "dtype": str(value.dtype),
            "device": str(value.device),
        }

    if isinstance(value, (tuple, list)):
        return [
            tensor_info(item)
            for item in value
            if isinstance(item, (torch.Tensor, tuple, list, dict))
        ]

    if isinstance(value, dict):
        return {
            key: tensor_info(item)
            for key, item in value.items()
            if isinstance(item, (torch.Tensor, tuple, list, dict))
        }

    return type(value).__name__


def register_debug_hooks(
    root,
    *,
    max_calls_per_module=1,
    leaf_only=True,
    name_filter=None,
):
    """
    root: hf_model, bridge или отдельный блок.
    max_calls_per_module: сколько вызовов каждого модуля печатать.
    leaf_only: печатать только модули без дочерних компонентов.
    name_filter: функция (name, module) -> bool.
    """
    handles = []
    counts = defaultdict(int)

    for name, module in root.named_modules():
        if not name:
            continue

        if leaf_only and any(module.children()):
            continue

        if name_filter is not None and not name_filter(name, module):
            continue

        module_name = name
        module_type = type(module).__name__

        def pre_hook(mod, args, kwargs, *, _name=module_name, _type=module_type):
            if counts[(_name, "pre")] >= max_calls_per_module:
                return

            counts[(_name, "pre")] += 1

            print(f"\n→ {_name} [{_type}]")
            print("  args:", tensor_info(args))

            if kwargs:
                print("  kwargs:", tensor_info(kwargs))

        def post_hook(
            mod,
            args,
            kwargs,
            output,
            *,
            _name=module_name,
            _type=module_type,
        ):
            if counts[(_name, "post")] >= max_calls_per_module:
                return

            counts[(_name, "post")] += 1
            print(f"← {_name} [{_type}]")
            print("  output:", tensor_info(output))

        handles.append(
            module.register_forward_pre_hook(
                pre_hook,
                with_kwargs=True,
            )
        )

        handles.append(
            module.register_forward_hook(
                post_hook,
                with_kwargs=True,
            )
        )

    print(f"Registered {len(handles)} hooks")
    return handles

In [16]:
handles = register_debug_hooks(
    bridge,
    max_calls_per_module=1,
    leaf_only=True,
)

Registered 54658 hooks


In [17]:
with torch.inference_mode():
    outputs, cache = bridge.run_with_cache(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
    )


→ embed.hook_in [HookPoint]
  args: [{'shape': (14, 33), 'dtype': 'torch.int64', 'device': 'cuda:0'}]
← embed.hook_in [HookPoint]
  output: {'shape': (14, 33), 'dtype': 'torch.int64', 'device': 'cuda:0'}

→ embed._original_component [Embedding]
  args: [{'shape': (14, 33), 'dtype': 'torch.int64', 'device': 'cuda:0'}]
← embed._original_component [Embedding]
  output: {'shape': (14, 33, 2048), 'dtype': 'torch.float16', 'device': 'cuda:0'}

→ embed.hook_out [HookPoint]
  args: [{'shape': (14, 33, 2048), 'dtype': 'torch.float16', 'device': 'cuda:0'}]
← embed.hook_out [HookPoint]
  output: {'shape': (14, 33, 2048), 'dtype': 'torch.float16', 'device': 'cuda:0'}

→ rotary_emb.hook_in [HookPoint]
  args: [{'shape': (14, 33, 2048), 'dtype': 'torch.float16', 'device': 'cuda:0'}]
← rotary_emb.hook_in [HookPoint]
  output: {'shape': (14, 33, 2048), 'dtype': 'torch.float16', 'device': 'cuda:0'}

→ rotary_emb._original_component [Qwen3MoeRotaryEmbedding]
  args: [{'shape': (14, 33, 2048), 'dtype': 